# Bankruptcy Prediction for Companies

Clean notebook for the empirical part of the thesis.

The notebook performs the following steps:
1. data loading and initial quality checks;
2. data cleaning;
3. feature engineering;
4. description of the final analytical sample;
5. company-level train-test split;
6. comparison of three Random Forest models:
   - financial features only;
   - due diligence index only;
   - financial features + due diligence index;
7. saving the final model;
8. prediction for a new company.


In [ ]:
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    RocCurveDisplay
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')


In [ ]:
# Mount Google Drive when the notebook is run in Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass


## 1. Loading the Source Dataset

In the source dataset, one row corresponds to one company in one reporting year.  
The target variable `target` takes the following values:

- `0` — operating company;
- `1` — bankrupt company.


In [ ]:
# =========================
# 1. DATA LOADING
# =========================

# The notebook checks several common locations for the input file.
# If the file is stored elsewhere, change the path manually here.
candidate_paths = [
    Path('/content/drive/MyDrive/thesis/массив объединенный.xlsx'),
    Path('/mnt/data/массив объединенный.xlsx')
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), candidate_paths[0])

df_raw = pd.read_excel(DATA_PATH)

# Standardize source column names:
# remove hidden characters and extra spaces
df_raw.columns = (
    df_raw.columns
    .str.replace('\ufeff', '', regex=False)
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Rename all source variables into English
column_rename_map = {
    'Регистрационный номер': 'registration_number',
    'Наименование': 'company_name',
    'Наименование на английском': 'company_name_english',
    'Краткое наименование': 'short_name',
    'Наименование полное': 'full_name',
    'Дата регистрации': 'registration_date',
    'Возраст компании, лет': 'company_age_years',
    'Код налогоплательщика': 'taxpayer_code',
    'Вид деятельности/отрасль': 'business_activity_industry',
    'Ранее использованные ИНН': 'previously_used_taxpayer_ids',
    'ИДО': 'due_diligence_index',
    'Сумма незавершенных исков в роли ответчика, RUB': 'pending_lawsuits_as_defendant_rub',
    'Исполнительные производства, RUB': 'enforcement_proceedings_rub',
    'Важная информация': 'important_information',
    'Активы всего': 'total_assets',
    'Внеоборотные активы': 'non_current_assets',
    'Выручка': 'revenue',
    'Дебиторская задолженность': 'accounts_receivable',
    'Денежные средства и денежные эквиваленты': 'cash_and_cash_equivalents',
    'Запасы': 'inventories',
    'Заёмные средства (долгосрочные)': 'long_term_borrowings',
    'Заёмные средства (краткосрочные)': 'short_term_borrowings',
    'Капитал и резервы': 'equity_and_reserves',
    'Краткосрочные обязательства': 'current_liabilities',
    'Краткосрочные финансовые вложения': 'short_term_financial_investments',
    'Кредиторская задолженность': 'accounts_payable',
    'Нематериальные активы': 'intangible_assets',
    'Нераспределенная прибыль (непокрытый убыток)': 'retained_earnings',
    'Оборотные активы': 'current_assets',
    'Прибыль (убыток) до налогообложения': 'profit_before_tax',
    'Прибыль (убыток) от продажи': 'profit_from_sales',
    'Проценты к уплате': 'interest_expense',
    'Себестоимость продаж': 'cost_of_sales',
    'Чистая прибыль (убыток)': 'net_profit'
}

df_raw = df_raw.rename(columns=column_rename_map)

print('Input file:', DATA_PATH)
print('Source dataset shape:', df_raw.shape)
display(df_raw.head())


## 2. Initial Data Quality Checks

This section checks the structure and quality of the source dataset before data cleaning and feature engineering.


In [ ]:
# =========================
# 2.1. GENERAL DATASET SUMMARY
# =========================

summary_table = pd.DataFrame({
    'Indicator': [
        'Number of rows',
        'Number of columns',
        'Number of unique companies',
        'Minimum year',
        'Maximum year',
        'Number of duplicate company-year pairs',
        'Number of companies with inconsistent target values across years'
    ],
    'Value': [
        len(df_raw),
        df_raw.shape[1],
        df_raw['registration_number'].nunique(),
        df_raw['year'].min(),
        df_raw['year'].max(),
        df_raw.duplicated(subset=['registration_number', 'year']).sum(),
        (df_raw.groupby('registration_number')['target'].nunique() > 1).sum()
    ]
})

display(summary_table)


In [ ]:
# =========================
# 2.2. MISSING VALUES IN THE SOURCE DATASET
# =========================

missing_table = pd.DataFrame({
    'Missing values': df_raw.isna().sum(),
    'Missing share, %': (df_raw.isna().mean() * 100).round(2)
})

missing_table = (
    missing_table[missing_table['Missing values'] > 0]
    .sort_values('Missing share, %', ascending=False)
)

display(missing_table)

top_missing = missing_table.head(15).sort_values('Missing share, %')

plt.figure(figsize=(10, 6))
plt.barh(top_missing.index, top_missing['Missing share, %'])
plt.title('Variables with the Highest Share of Missing Values')
plt.xlabel('Missing share, %')
plt.tight_layout()
plt.show()


## 3. Data Cleaning

Before financial ratios are calculated, observations with clearly invalid values are removed:

- negative revenue;
- non-positive total assets;
- non-positive current liabilities;
- negative accounts receivable;
- negative accounts payable.


In [ ]:
# =========================
# 3. DATA CLEANING
# =========================

df_clean = df_raw.copy()

mask_invalid = (
    (df_clean['revenue'] < 0) |
    (df_clean['total_assets'] <= 0) |
    (df_clean['current_liabilities'] <= 0) |
    (df_clean['accounts_receivable'] < 0) |
    (df_clean['accounts_payable'] < 0)
)

invalid_summary = pd.DataFrame({
    'Rule': [
        'Revenue < 0',
        'Total assets <= 0',
        'Current liabilities <= 0',
        'Accounts receivable < 0',
        'Accounts payable < 0',
        'Unique rows removed by at least one rule'
    ],
    'Number of rows': [
        (df_clean['revenue'] < 0).sum(),
        (df_clean['total_assets'] <= 0).sum(),
        (df_clean['current_liabilities'] <= 0).sum(),
        (df_clean['accounts_receivable'] < 0).sum(),
        (df_clean['accounts_payable'] < 0).sum(),
        mask_invalid.sum()
    ]
})

display(invalid_summary)

df_clean = df_clean.loc[~mask_invalid].copy()

print('Dataset shape before cleaning:', df_raw.shape)
print('Dataset shape after cleaning:', df_clean.shape)


## 4. Feature Engineering

Financial ratios are calculated from absolute financial statement items to describe profitability, liquidity, turnover, financial stability, company size, and company age.


In [ ]:
# =========================
# 4. FEATURE ENGINEERING
# =========================

def safe_divide(numerator, denominator):
    return np.where(denominator != 0, numerator / denominator, np.nan)

df_features = df_clean.copy()

# Profitability
df_features['roa'] = safe_divide(
    df_features['net_profit'],
    df_features['total_assets']
)

df_features['roe'] = safe_divide(
    df_features['net_profit'],
    df_features['equity_and_reserves']
)

df_features['profitability_of_sales'] = safe_divide(
    df_features['net_profit'],
    df_features['revenue']
)

df_features['operating_margin'] = safe_divide(
    df_features['profit_from_sales'],
    df_features['revenue']
)

# Liquidity and turnover
df_features['current_ratio'] = safe_divide(
    df_features['current_assets'],
    df_features['current_liabilities']
)

df_features['receivables_turnover'] = safe_divide(
    df_features['revenue'],
    df_features['accounts_receivable']
)

df_features['payables_turnover'] = safe_divide(
    df_features['cost_of_sales'],
    df_features['accounts_payable']
)

df_features['asset_turnover'] = safe_divide(
    df_features['revenue'],
    df_features['total_assets']
)

# Financial stability
df_features['autonomy_ratio'] = safe_divide(
    df_features['equity_and_reserves'],
    df_features['total_assets']
)

df_features['retained_earnings_to_revenue'] = safe_divide(
    df_features['retained_earnings'],
    df_features['revenue']
)

# Company size and age
df_features['log_assets'] = np.log1p(df_features['total_assets'])
df_features['company_age'] = df_features['company_age_years']

financial_features = [
    'roa',
    'roe',
    'profitability_of_sales',
    'operating_margin',
    'current_ratio',
    'receivables_turnover',
    'payables_turnover',
    'asset_turnover',
    'autonomy_ratio',
    'retained_earnings_to_revenue',
    'log_assets',
    'company_age'
]

behavioral_features = ['due_diligence_index']
combined_features = financial_features + behavioral_features

feature_description = pd.DataFrame({
    'Feature': financial_features + behavioral_features,
    'Group': ['Financial'] * len(financial_features) + ['Behavioral'],
    'Interpretation': [
        'Return on assets',
        'Return on equity',
        'Profitability of sales',
        'Operating margin',
        'Current ratio',
        'Accounts receivable turnover',
        'Accounts payable turnover',
        'Asset turnover',
        'Autonomy ratio',
        'Retained earnings to revenue',
        'Logarithm of total assets',
        'Company age',
        'Due diligence index'
    ]
})

display(feature_description)


## 5. Description of the Final Analytical Sample

After invalid observations are removed and model features are constructed, the final analytical sample is formed for model training and testing. The section describes its size, annual structure, class distribution, missing values, and descriptive statistics for model features.


In [ ]:
# =========================
# 5. DESCRIPTION OF THE FINAL ANALYTICAL SAMPLE
# =========================

company_col = 'registration_number'
target_col = 'target'

# ---------------------------------------------------------
# 1. General characteristics of the final sample
# ---------------------------------------------------------
final_sample_overview = pd.DataFrame({
    'Indicator': [
        'Number of observations',
        'Number of unique companies',
        'Observation period',
        'Number of years',
        'Number of observations for operating companies',
        'Number of observations for bankrupt companies',
        'Share of observations for bankrupt companies, %'
    ],
    'Value': [
        len(df_features),
        df_features[company_col].nunique(),
        f"{int(df_features['year'].min())}–{int(df_features['year'].max())}",
        df_features['year'].nunique(),
        int((df_features[target_col] == 0).sum()),
        int((df_features[target_col] == 1).sum()),
        round(df_features[target_col].mean() * 100, 2)
    ]
})

print('General characteristics of the final analytical sample:')
display(final_sample_overview)

# ---------------------------------------------------------
# 2. Distribution of observations by year and class
# ---------------------------------------------------------
year_class_summary = pd.crosstab(
    df_features['year'],
    df_features[target_col]
).reindex(columns=[0, 1], fill_value=0)

year_class_summary.columns = [
    'Operating companies',
    'Bankrupt companies'
]

year_class_summary['Total observations'] = year_class_summary.sum(axis=1)
year_class_summary['Share of bankrupt companies, %'] = (
    year_class_summary['Bankrupt companies'] /
    year_class_summary['Total observations'] * 100
).round(2)

print('Distribution of observations by year:')
display(year_class_summary)

# Figure 1. Total number of observations by year
plt.figure(figsize=(8, 5))
plt.bar(
    year_class_summary.index.astype(str),
    year_class_summary['Total observations']
)
plt.title('Distribution of Observations by Year')
plt.xlabel('Year')
plt.ylabel('Number of observations')
plt.show()

# Figure 2. Distribution of operating and bankrupt companies by year
year_class_summary[
    ['Operating companies', 'Bankrupt companies']
].plot(
    kind='bar',
    stacked=True,
    figsize=(8, 5)
)
plt.title('Class Distribution by Year')
plt.xlabel('Year')
plt.ylabel('Number of observations')
plt.legend(title='Class')
plt.show()

# Figure 3. Share of bankrupt companies by year
plt.figure(figsize=(8, 5))
plt.plot(
    year_class_summary.index,
    year_class_summary['Share of bankrupt companies, %'],
    marker='o'
)
plt.title('Share of Bankrupt Companies by Year')
plt.xlabel('Year')
plt.ylabel('Share of bankrupt companies, %')
plt.xticks(year_class_summary.index)
plt.show()

# ---------------------------------------------------------
# 3. Class distribution in the final sample
# ---------------------------------------------------------
class_summary = (
    df_features[target_col]
    .value_counts()
    .sort_index()
    .rename_axis('target')
    .reset_index(name='Number of observations')
)

class_summary['Class'] = class_summary['target'].map({
    0: 'Operating companies',
    1: 'Bankrupt companies'
})

class_summary['Share, %'] = (
    class_summary['Number of observations'] /
    len(df_features) * 100
).round(2)

class_summary = class_summary[
    ['Class', 'Number of observations', 'Share, %']
]

print('Class distribution in the final sample:')
display(class_summary)

plt.figure(figsize=(7, 5))
plt.bar(
    class_summary['Class'],
    class_summary['Number of observations']
)
plt.title('Class Distribution in the Final Sample')
plt.ylabel('Number of observations')
plt.xticks(rotation=0)
plt.show()

# ---------------------------------------------------------
# 4. Missing values in final model features
# ---------------------------------------------------------
missing_features_summary = pd.DataFrame({
    'Feature': combined_features,
    'Missing values': df_features[combined_features].isna().sum().values,
    'Missing share, %': (
        df_features[combined_features].isna().mean().values * 100
    ).round(2)
}).sort_values('Missing share, %', ascending=False)

print('Missing values in final model features:')
display(missing_features_summary)

missing_features_to_plot = missing_features_summary[
    missing_features_summary['Missing values'] > 0
]

if not missing_features_to_plot.empty:
    plt.figure(figsize=(10, 6))
    plt.barh(
        missing_features_to_plot['Feature'],
        missing_features_to_plot['Missing share, %']
    )
    plt.title('Missing Values in Model Features')
    plt.xlabel('Missing share, %')
    plt.gca().invert_yaxis()
    plt.show()
else:
    print('There are no missing values in the final model features.')

# ---------------------------------------------------------
# 5. Descriptive statistics for final model features
# ---------------------------------------------------------
descriptive_statistics = df_features[combined_features].describe().T

print('Descriptive statistics for model features:')
display(descriptive_statistics)

# ---------------------------------------------------------
# 6. Median feature values by class
# ---------------------------------------------------------
median_by_class = (
    df_features
    .groupby(target_col)[combined_features]
    .median()
    .T
)

median_by_class.columns = [
    'Median: operating companies',
    'Median: bankrupt companies'
]

print('Median feature values by class:')
display(median_by_class)

# ---------------------------------------------------------
# 7. Short automated summary of the final sample
# ---------------------------------------------------------
min_year_share = year_class_summary['Total observations'].min() / len(df_features) * 100
max_year_share = year_class_summary['Total observations'].max() / len(df_features) * 100

print(
    f"The final analytical sample includes {len(df_features):,} observations "
    f"for {df_features[company_col].nunique():,} unique companies over "
    f"{int(df_features['year'].min())}–{int(df_features['year'].max())}. "
    f"The annual share of observations ranges from {min_year_share:.2f}% "
    f"to {max_year_share:.2f}% of the total sample. "
    f"The share of observations for bankrupt companies is "
    f"{df_features[target_col].mean() * 100:.2f}%."
)


## 6. Train-Test Split

The split is performed at the **company level**.  
Each company is assigned either to the training set or to the test set, so observations of the same company from different years cannot appear in both subsets.


In [ ]:
# =========================
# 6. COMPANY-LEVEL TRAIN-TEST SPLIT
# =========================

company_col = 'registration_number'
target_col = 'target'

company_target = (
    df_features
    .groupby(company_col)[target_col]
    .first()
)

train_companies, test_companies = train_test_split(
    company_target.index,
    test_size=0.2,
    random_state=42,
    stratify=company_target
)

train_idx = df_features.index[df_features[company_col].isin(train_companies)]
test_idx = df_features.index[df_features[company_col].isin(test_companies)]

split_summary = pd.DataFrame({
    'Sample subset': ['Train', 'Test'],
    'Number of companies': [len(train_companies), len(test_companies)],
    'Number of observations': [len(train_idx), len(test_idx)],
    'Share of bankrupt companies, %': [
        round(df_features.loc[train_idx, target_col].mean() * 100, 2),
        round(df_features.loc[test_idx, target_col].mean() * 100, 2)
    ]
})

display(split_summary)

print(
    'Number of overlapping companies between train and test:',
    len(set(train_companies) & set(test_companies))
)


## 7. Model Preprocessing

For each model, preprocessing is performed separately:

1. extreme values are clipped at the 1st and 99th percentiles calculated on the **training set only**;
2. missing values are imputed with medians calculated on the **training set only**.

The test set does not influence preprocessing parameters.


In [ ]:
# =========================
# 7. MODEL PREPROCESSING FUNCTION
# =========================

def prepare_train_test(df, train_idx, test_idx, features, winsorize_features=None):
    X_train = df.loc[train_idx, features].copy()
    X_test = df.loc[test_idx, features].copy()

    if winsorize_features is None:
        winsorize_features = []

    winsor_bounds_local = {}

    # Winsorization bounds are calculated on the training set only
    for col in features:
        if col in winsorize_features:
            lower = X_train[col].quantile(0.01)
            upper = X_train[col].quantile(0.99)

            winsor_bounds_local[col] = (lower, upper)

            X_train[col] = X_train[col].clip(lower=lower, upper=upper)
            X_test[col] = X_test[col].clip(lower=lower, upper=upper)

    # Missing values are imputed using training-set medians only
    imputer_local = SimpleImputer(strategy='median')

    X_train_imputed = pd.DataFrame(
        imputer_local.fit_transform(X_train),
        columns=features,
        index=X_train.index
    )

    X_test_imputed = pd.DataFrame(
        imputer_local.transform(X_test),
        columns=features,
        index=X_test.index
    )

    return X_train_imputed, X_test_imputed, imputer_local, winsor_bounds_local


## 8. Benchmarking Alternative Algorithms

After the analytical sample has been transformed and the train-test split has been fixed, several alternative algorithms are estimated on the same combined set of predictors. This block is used only as an empirical robustness check. It does not change the subsequent final model: the deployed specification remains the Random Forest model with financial features and the due diligence index.


In [ ]:
# =========================
# 8. BENCHMARKING ALTERNATIVE ALGORITHMS
# =========================

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# The benchmark uses the same train-test split and the same combined feature set
# as the final model. Preprocessing remains identical: winsorization is estimated
# on the training set only, followed by median imputation.
X_train_benchmark, X_test_benchmark, _, _ = prepare_train_test(
    df_features,
    train_idx,
    test_idx,
    combined_features,
    winsorize_features=[col for col in combined_features if col in financial_features]
)

y_train_benchmark = df_features.loc[train_idx, target_col]
y_test_benchmark = df_features.loc[test_idx, target_col]

benchmark_models = {
    'Linear regression (linear probability baseline)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Logistic regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            max_iter=2000,
            class_weight='balanced',
            random_state=42
        ))
    ]),
    'Decision tree': DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=10,
        random_state=42,
        class_weight='balanced'
    ),
    'Gradient boosting': GradientBoostingClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ),
    'Random forest': RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )
}

benchmark_results = []

for benchmark_name, benchmark_model in benchmark_models.items():
    benchmark_model.fit(X_train_benchmark, y_train_benchmark)

    if hasattr(benchmark_model, 'predict_proba'):
        y_score_benchmark = benchmark_model.predict_proba(X_test_benchmark)[:, 1]
    else:
        # Linear regression is treated as a linear probability baseline.
        # Its continuous predictions are clipped to the [0, 1] interval.
        y_score_benchmark = np.clip(
            benchmark_model.predict(X_test_benchmark),
            0,
            1
        )

    y_pred_benchmark = (y_score_benchmark >= 0.5).astype(int)

    benchmark_results.append({
        'Model': benchmark_name,
        'accuracy': accuracy_score(y_test_benchmark, y_pred_benchmark),
        'precision': precision_score(y_test_benchmark, y_pred_benchmark, zero_division=0),
        'recall': recall_score(y_test_benchmark, y_pred_benchmark, zero_division=0),
        'f1_score': f1_score(y_test_benchmark, y_pred_benchmark, zero_division=0),
        'roc_auc': roc_auc_score(y_test_benchmark, y_score_benchmark)
    })

benchmark_results_df = (
    pd.DataFrame(benchmark_results)
    .sort_values('roc_auc', ascending=False)
    .reset_index(drop=True)
)

display(benchmark_results_df.style.format({
    'accuracy': '{:.4f}',
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1_score': '{:.4f}',
    'roc_auc': '{:.4f}'
}))

best_benchmark_model = benchmark_results_df.iloc[0]['Model']
print(f"Best model in the algorithm benchmark by ROC-AUC: {best_benchmark_model}")

rf_benchmark_row = benchmark_results_df.loc[
    benchmark_results_df['Model'] == 'Random forest'
].iloc[0]

print(
    "The Random Forest model is retained for the next stage because it provides "
    f"a strong balance of ROC-AUC ({rf_benchmark_row['roc_auc']:.4f}), "
    f"recall ({rf_benchmark_row['recall']:.4f}) and F1-score "
    f"({rf_benchmark_row['f1_score']:.4f}), while also allowing feature importance analysis."
)


## 9. Random Forest Specification Comparison

After the general algorithm benchmark, three Random Forest specifications are compared on the same test set:

1. **Financial features only**;
2. **Due diligence index only**;
3. **Financial features + due diligence index**.


In [ ]:
# =========================
# 9. RANDOM FOREST SPECIFICATION COMPARISON
# =========================

feature_sets = {
    'Financial features only': financial_features,
    'Due diligence index only': behavioral_features,
    'Financial features + due diligence index': combined_features
}

results = []
trained_models = {}
trained_imputers = {}
trained_winsor_bounds = {}
prepared_test_sets = {}
confusion_matrices = {}

for model_name, features in feature_sets.items():

    X_train_local, X_test_local, imputer_local, winsor_bounds_local = prepare_train_test(
        df_features,
        train_idx,
        test_idx,
        features,
        winsorize_features=[col for col in features if col in financial_features]
    )

    y_train_local = df_features.loc[train_idx, target_col]
    y_test_local = df_features.loc[test_idx, target_col]

    model = RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    )

    model.fit(X_train_local, y_train_local)

    y_pred_local = model.predict(X_test_local)
    y_proba_local = model.predict_proba(X_test_local)[:, 1]

    results.append({
        'Model': model_name,
        'accuracy': accuracy_score(y_test_local, y_pred_local),
        'precision': precision_score(y_test_local, y_pred_local, zero_division=0),
        'recall': recall_score(y_test_local, y_pred_local, zero_division=0),
        'f1_score': f1_score(y_test_local, y_pred_local, zero_division=0),
        'roc_auc': roc_auc_score(y_test_local, y_proba_local)
    })

    trained_models[model_name] = model
    trained_imputers[model_name] = imputer_local
    trained_winsor_bounds[model_name] = winsor_bounds_local
    prepared_test_sets[model_name] = (X_test_local, y_test_local, y_proba_local)
    confusion_matrices[model_name] = confusion_matrix(y_test_local, y_pred_local)

results_df = (
    pd.DataFrame(results)
    .sort_values('roc_auc', ascending=False)
    .reset_index(drop=True)
)

display(results_df.style.format({
    'accuracy': '{:.4f}',
    'precision': '{:.4f}',
    'recall': '{:.4f}',
    'f1_score': '{:.4f}',
    'roc_auc': '{:.4f}'
}))


In [ ]:
# =========================
# 9.1. ROC CURVES FOR THE THREE RANDOM FOREST SPECIFICATIONS
# =========================

plt.figure(figsize=(7, 6))

for model_name in feature_sets:
    X_test_local, y_test_local, y_proba_local = prepared_test_sets[model_name]
    RocCurveDisplay.from_predictions(
        y_test_local,
        y_proba_local,
        name=model_name,
        ax=plt.gca()
    )

plt.title('ROC Curves for the Compared Models')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report

final_model_name = 'Financial features + due diligence index'

# The final model uses the combined set of features
final_features = combined_features

# Prepare train and test sets for the final model
X_train_final, X_test_final, _, _ = prepare_train_test(
    df_features,
    train_idx,
    test_idx,
    final_features,
    winsorize_features=[col for col in final_features if col in financial_features]
)

y_test_final = df_features.loc[test_idx, target_col]

# Prediction for the final model
y_pred_final = trained_models[final_model_name].predict(X_test_final)

print("Classification Report:")
print(
    classification_report(
        y_test_final,
        y_pred_final,
        target_names=["Non-bankrupt companies", "Bankrupt companies"],
        digits=4
    )
)


In [ ]:
# =========================
# 9.2. AUTOMATED HYPOTHESIS SUMMARY
# =========================

financial_row = results_df.loc[results_df['Model'] == 'Financial features only'].iloc[0]
combined_row = results_df.loc[
    results_df['Model'] == 'Financial features + due diligence index'
].iloc[0]

print('Comparison of the financial and combined models:')
print(f"ROC-AUC: {financial_row['roc_auc']:.4f} → {combined_row['roc_auc']:.4f}")
print(f"Recall:  {financial_row['recall']:.4f} → {combined_row['recall']:.4f}")
print(f"F1-score:{financial_row['f1_score']:.4f} → {combined_row['f1_score']:.4f}")

if combined_row['roc_auc'] > financial_row['roc_auc']:
    print('\nThe addition of the due diligence index increases predictive performance relative to the model based on financial features only.')
else:
    print('\nThe addition of the due diligence index does not increase ROC-AUC relative to the model based on financial features only.')


## 10. Final Model

The final specification is the combined model **Financial features + due diligence index**.


In [ ]:
# =========================
# 10. FINAL MODEL
# =========================

final_model_name = 'Financial features + due diligence index'

rf = trained_models[final_model_name]
imputer = trained_imputers[final_model_name]
winsor_bounds = trained_winsor_bounds[final_model_name]
final_features = combined_features.copy()

X_test_final, y_test_final, y_proba_final = prepared_test_sets[final_model_name]
y_pred_final = rf.predict(X_test_final)

cm = confusion_matrices[final_model_name]

cm_df = pd.DataFrame(
    cm,
    index=['Actual 0', 'Actual 1'],
    columns=['Predicted 0', 'Predicted 1']
)

display(cm_df)

feature_importance = pd.DataFrame({
    'Feature': final_features,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

display(feature_importance)

plt.figure(figsize=(8, 6))
plt.barh(
    feature_importance['Feature'][::-1],
    feature_importance['Importance'][::-1]
)
plt.title('Feature Importance in the Final Model')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()


## 11. Saving the Final Model

The following objects are saved in one file:

- trained model;
- final feature list;
- imputer;
- winsorization bounds;
- list of raw input fields required for feature construction.


In [ ]:
# =========================
# 11. SAVING THE FINAL MODEL
# =========================

raw_input_cols = [
    'revenue',
    'total_assets',
    'net_profit',
    'equity_and_reserves',
    'profit_from_sales',
    'current_assets',
    'current_liabilities',
    'accounts_receivable',
    'cost_of_sales',
    'accounts_payable',
    'retained_earnings',
    'company_age_years',
    'due_diligence_index'
]

model_bundle = {
    'model': rf,
    'imputer': imputer,
    'features': final_features,
    'winsor_bounds': winsor_bounds,
    'raw_input_cols': raw_input_cols,
    'target_definition': {
        0: 'Operating company',
        1: 'Bankrupt company'
    },
    'threshold': 0.5
}

MODEL_PATH = 'bankruptcy_rf_bundle_clean.pkl'
joblib.dump(model_bundle, MODEL_PATH)

print(f'Final model saved to: {MODEL_PATH}')


## 11. Functions for New-Company Prediction


In [ ]:
# =========================
# 12. FUNCTIONS FOR NEW-COMPANY PREDICTION
# =========================

def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace('\ufeff', '', regex=False)
        .str.replace('\xa0', ' ', regex=False)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )
    return df

def build_features_for_prediction(df_input):
    df_input = normalize_columns(df_input)

    df_out = df_input.copy()

    df_out['roa'] = safe_divide(
        df_out['net_profit'],
        df_out['total_assets']
    )

    df_out['roe'] = safe_divide(
        df_out['net_profit'],
        df_out['equity_and_reserves']
    )

    df_out['profitability_of_sales'] = safe_divide(
        df_out['net_profit'],
        df_out['revenue']
    )

    df_out['operating_margin'] = safe_divide(
        df_out['profit_from_sales'],
        df_out['revenue']
    )

    df_out['current_ratio'] = safe_divide(
        df_out['current_assets'],
        df_out['current_liabilities']
    )

    df_out['receivables_turnover'] = safe_divide(
        df_out['revenue'],
        df_out['accounts_receivable']
    )

    df_out['payables_turnover'] = safe_divide(
        df_out['cost_of_sales'],
        df_out['accounts_payable']
    )

    df_out['asset_turnover'] = safe_divide(
        df_out['revenue'],
        df_out['total_assets']
    )

    df_out['autonomy_ratio'] = safe_divide(
        df_out['equity_and_reserves'],
        df_out['total_assets']
    )

    df_out['retained_earnings_to_revenue'] = safe_divide(
        df_out['retained_earnings'],
        df_out['revenue']
    )

    df_out['log_assets'] = np.log1p(df_out['total_assets'])
    df_out['company_age'] = df_out['company_age_years']

    return df_out

def predict_companies(df_new, bundle_path=MODEL_PATH):
    bundle = joblib.load(bundle_path)

    model = bundle['model']
    imputer_loaded = bundle['imputer']
    features_loaded = bundle['features']
    winsor_bounds_loaded = bundle['winsor_bounds']
    threshold = bundle['threshold']

    df_new_features = build_features_for_prediction(df_new)
    X_new = df_new_features[features_loaded].copy()

    for col, (lower, upper) in winsor_bounds_loaded.items():
        X_new[col] = X_new[col].clip(lower=lower, upper=upper)

    X_new_imputed = pd.DataFrame(
        imputer_loaded.transform(X_new),
        columns=features_loaded,
        index=X_new.index
    )

    bankruptcy_probability = model.predict_proba(X_new_imputed)[:, 1]
    predicted_class = (bankruptcy_probability >= threshold).astype(int)

    result = df_new.copy()
    result['bankruptcy_probability'] = bankruptcy_probability
    result['predicted_class'] = predicted_class
    result['risk_category'] = pd.cut(
        bankruptcy_probability,
        bins=[-np.inf, 0.30, 0.70, np.inf],
        labels=['Low', 'Medium', 'High']
    )

    return result


## 12. Prediction Example for a New Company


In [ ]:
# =========================
# 12. PREDICTION EXAMPLE
# =========================

new_company = pd.DataFrame({
    'revenue': [500_000_000],
    'total_assets': [450_000_000],
    'net_profit': [25_000_000],
    'equity_and_reserves': [180_000_000],
    'profit_from_sales': [40_000_000],
    'current_assets': [220_000_000],
    'current_liabilities': [130_000_000],
    'accounts_receivable': [80_000_000],
    'cost_of_sales': [360_000_000],
    'accounts_payable': [70_000_000],
    'retained_earnings': [60_000_000],
    'company_age_years': [8],
    'due_diligence_index': [85]
})

prediction_example = predict_companies(new_company)
display(prediction_example)
